# 🐷 Pig Posture Recognition – Advanced Kaggle Strategies

Dieses Notebook testet drei vielversprechende Konzepte der Forschung und aktueller Kaggle-Setups:
1. **Deep Separable Convolutions:** MobileNetV3 (effizient, wenig Overfitting bei "schmutzigen" Stallbildern)
2. **Hintergrundunterdrückung (Thresholding/Blur):** Ein Pre-Processing-Schritt, der lenkenden Stallhintergrund (Gitter, Stroh) abschwächt.
3. **Focal Loss:** Bestraft algorithmisch den "Lying-Bias" (das Modell rät oft "Liegend", weil das statistisch häufiger vorkommt).

## ⚙️ Configuration

In [12]:
TAG = "T2"   # "T1" or "T2"

DATA_ROOT = "/datasets/multi-view-pig-posture-recognition"

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/advanced_strategies_{TAG.lower()}"

# Model: MobileNetV3 (Deep Separable Convolutions)
MODEL_NAME     = "mobilenetv3_large_100" 
IMG_SIZE       = 224
BATCH_SIZE     = 64
EPOCHS         = 20
LR             = 1e-3
VAL_FRAC       = 0.10
PAD_RATIO      = 0.15
NUM_WORKERS    = 8
SEED           = 42
NUM_CLASSES    = 5

CLASS_NAMES    = ["Lateral_lying_left", "Lateral_lying_right",
                  "Sitting", "Standing", "Sternal_lying"]

print(f"Tag: {TAG}  |  Model: {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")

Tag: T2  |  Model: mobilenetv3_large_100
Output: runs/advanced_strategies_t2


## 📦 Install Dependencies

In [2]:
!pip install timm tqdm scikit-learn pandas pillow matplotlib opencv-python -q

  error: subprocess-exited-with-error
  
  × installing build dependencies for opencv-python did not run successfully.
  │ exit code: 1
  ╰─> [47 lines of output]
      Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
      Ignoring numpy: markers 'python_version == "3.13"' don't match your environment
      Ignoring setuptools: markers 'python_version >= "3.12"' don't match your environment
        Using cached numpy-2.0.2.tar.gz (18.9 MB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Installing backend dependencies: started
        Installing backend dependencies: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'error'
        error: subprocess-exited-with-err

## 📚 Imports

In [13]:
import os, ast, random
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import cv2
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 🔒 Reproducibility

In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1️⃣ Background Blur Pre-Processing
Eine "Light"-Version der Hintergrund-Eliminierung. Wir weichzeichnen die Randbereiche des Boundingbox-Crops, um den Fokus auf die Bildmitte (das Schwein) zu legen.

In [5]:
def apply_vignette_blur(img: Image.Image, blur_radius=5) -> Image.Image:
    """
    Wendet einen starken Blur auf die Bildränder an, lässt das Zentrum scharf.
    Hilft dem Netzwerk, Gitterstäbe am Rand zu ignorieren.
    """
    # Erstelle eine Maske (Zentrum weiß, Ränder schwarz)
    w, h = img.size
    mask = Image.new('L', (w, h), 0)
    from PIL import ImageDraw
    draw = ImageDraw.Draw(mask)
    # Wir zeichnen eine Ellipse im Zentrum, die scharf bleibt
    margin_w, margin_h = int(w * 0.15), int(h * 0.15)
    draw.ellipse((margin_w, margin_h, w - margin_w, h - margin_h), fill=255)
    # Maske extrem weichzeichnen für sanften Übergang
    mask = mask.filter(ImageFilter.GaussianBlur(min(w, h) * 0.1))
    
    # Erstelle ein komplett verschwommenes Bild
    blurred_img = img.filter(ImageFilter.GaussianBlur(blur_radius))
    
    # Kombiniere (Zentrum vom Original, Ränder vom Blurred)
    return Image.composite(img, blurred_img, mask)

## 🗂️ Dataset (mit Blur optional)

In [6]:
df = pd.read_csv(CSV_PATH)

class PigPostureDatasetAdvanced(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.15, use_bg_blur=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.use_bg_blur = use_bg_blur

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        
        if self.use_bg_blur:
            crop = apply_vignette_blur(crop)

        if self.transform: crop = self.transform(crop)
        return crop, int(row["class_id"])

def get_train_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size + 32, size + 32)),
        T.RandomCrop(size),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.3, contrast=0.3),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

def get_val_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size, size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


## ✂️ Train / Validation Split

In [7]:
train_df, val_df = train_test_split(
    df, test_size=VAL_FRAC, stratify=df["class_id"], random_state=SEED
)

# Wir testen hier den Blur auf den Trainingsdaten
train_ds = PigPostureDatasetAdvanced(train_df, IMG_DIR, transform=get_train_transform(), use_bg_blur=True)
val_ds   = PigPostureDatasetAdvanced(val_df,   IMG_DIR, transform=get_val_transform(), use_bg_blur=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

## 2️⃣ Focal Loss
Klassische Cross-Entropy überschüttert das Modell bei starkem Klassen-Imbalance oft mit Gradients der Mehrheitsklassen. Focal Loss reduziert das Gewicht für Beispiele, die das Modell bereits mit hoher Wahrscheinlichkeit richtig vorhersagt ("easy examples"), und zwingt das Netzwerk, sich auf harte Fehler (z.B. Stand vs. Lying) zu konzentrieren.

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        if alpha is not None:
            if isinstance(alpha, (float, int)):
                self.alpha = torch.Tensor([alpha, 1-alpha])
            elif isinstance(alpha, list):
                self.alpha = torch.Tensor(alpha)
            else:
                self.alpha = alpha
        else:
            self.alpha = None

    def forward(self, input, target):
        if input.dim() > 2:
            input = input.view(input.size(0), input.size(1), -1)  # N,C,H,W => N,C,H*W
            input = input.transpose(1, 2)    # N,C,H*W => N,H*W,C
            input = input.contiguous().view(-1, input.size(2))   # N,H*W,C => N*H*W,C
        target = target.view(-1, 1)

        logpt = F.log_softmax(input, dim=1)
        logpt = logpt.gather(1, target)
        logpt = logpt.view(-1)
        pt = logpt.detach().exp()

        if self.alpha is not None:
            self.alpha = self.alpha.to(input.device)
            at = self.alpha.gather(0, target.view(-1))
            logpt = logpt * at

        loss = -1 * (1 - pt)**self.gamma * logpt
        return loss.mean()

## 3️⃣ Deep Separable Convolutions (MobileNetV3)
Weniger Parameter bedeuten hier in der Theorie weniger Overfitting auf den Käfighintergrund.

In [9]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {MODEL_NAME}  |  Parameters: {total_params:.1f}M")

# Class-weighted Focal Loss
counts  = Counter(train_df["class_id"].tolist())
total   = len(train_df)
weights = torch.tensor(
    [total / (NUM_CLASSES * max(counts.get(c, 1), 1)) for c in range(NUM_CLASSES)],
    dtype=torch.float32
).to(DEVICE)
print("Class weights (Alpha für Focal Loss):", [f"{w:.2f}" for w in weights.cpu()])

criterion = FocalLoss(alpha=weights, gamma=2.0)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

Model: mobilenetv3_large_100  |  Parameters: 4.2M
Class weights (Alpha für Focal Loss): ['1.52', '1.37', '6.74', '0.47', '0.74']


## 🛠️ Training Helpers

In [10]:
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0)

@torch.no_grad()
def validate_epoch(model, loader):
    model.eval()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0), preds_all, labels_all

## 🚀 Training Loop

In [14]:
best_val_f1 = 0.0
log         = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1       = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss, val_f1, _, _     = validate_epoch(model, val_loader)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    mark = "★" if val_f1 > best_val_f1 else " "
    print(f"{mark} Epoch {epoch:03d}/{EPOCHS} | "
          f"Train loss={train_loss:.4f} f1={train_f1:.4f} | "
          f"Val loss={val_loss:.4f} f1={val_f1:.4f} | lr={lr:.2e}")

    log.append(dict(epoch=epoch, train_loss=train_loss, train_f1=train_f1,
                    val_loss=val_loss, val_f1=val_f1, lr=lr))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        ckpt_path   = os.path.join(OUTPUT_DIR, "best_model.pth")
        torch.save({
            "epoch": epoch, "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_f1": val_f1, "model_name": MODEL_NAME, "tag": TAG,
        }, ckpt_path)
        print(f"  → Saved best model (val_f1={val_f1:.4f})")

print(f"\n✅ Training complete. Best Val F1: {best_val_f1:.4f}")
pd.DataFrame(log).to_csv(os.path.join(OUTPUT_DIR, "training_log.csv"), index=False)

★ Epoch 001/20 | Train loss=0.6871 f1=0.5783 | Val loss=0.3785 f1=0.6563 | lr=9.98e-04
  → Saved best model (val_f1=0.6563)


  Epoch 002/20 | Train loss=0.3500 f1=0.7176 | Val loss=0.4980 f1=0.5621 | lr=9.94e-04


★ Epoch 003/20 | Train loss=0.2031 f1=0.7879 | Val loss=0.2489 f1=0.8091 | lr=9.86e-04
  → Saved best model (val_f1=0.8091)


★ Epoch 004/20 | Train loss=0.1362 f1=0.8383 | Val loss=0.2129 f1=0.8594 | lr=9.76e-04
  → Saved best model (val_f1=0.8594)


  Epoch 005/20 | Train loss=0.1022 f1=0.8712 | Val loss=0.2970 f1=0.8166 | lr=9.62e-04


...

★ Epoch 016/20 | Train loss=0.0364 f1=0.9510 | Val loss=0.2664 f1=0.8844 | lr=6.55e-04
  → Saved best model (val_f1=0.8844)


★ Epoch 017/20 | Train loss=0.0228 f1=0.9658 | Val loss=0.2895 f1=0.8895 | lr=6.17e-04
  → Saved best model (val_f1=0.8895)


  Epoch 018/20 | Train loss=0.0248 f1=0.9640 | Val loss=0.2783 f1=0.8521 | lr=5.79e-04


  Epoch 019/20 | Train loss=0.0188 f1=0.9734 | Val loss=0.2558 f1=0.8648 | lr=5.40e-04


  Epoch 020/20 | Train loss=0.0196 f1=0.9716 | Val loss=0.4041 f1=0.8873 | lr=5.00e-04

✅ Training complete. Best Val F1: 0.8895


## 📊 Evaluation
Macht dieses Setup Sinn? Hier vergleichen wir die Ergebnisse der speziellen Strategien (MobileNetV3 + Blur + Focal Loss) mit dem Klassiker (DenseNet/ConvNeXt + CrossEntropy).
Wenn die Accuracy deutlich steigt, besonders bei seltenen Klassen wie 'Standing', hat Focal Loss und Blur angeschlagen.

In [ ]:
best_ckpt = torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), map_location=DEVICE)
model.load_state_dict(best_ckpt["model"])
_, _, val_preds, val_labels = validate_epoch(model, val_loader)
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES))